# Pipeline Preprocessing Terintegrasi
## Geopolitics News × BI USD Rate

Notebook ini mengimplementasikan pipeline preprocessing untuk menggabungkan dataset **berita geopolitik** (CNBC, kolom: `keyword_matched, title, url, preview_summary, full_text, date_raw, section`) dengan **data kurs USD Bank Indonesia** (`bi-usd-rate.csv`), untuk keperluan *time-series forecasting* / *sentiment-driven financial modeling*.

**Struktur pipeline:**
1. Setup & konfigurasi
2. Load data mentah
3. Filtering data berita geopolitik
4. Cleaning data kurs BI
5. Text preprocessing (NLP pipeline)
6. Feature engineering kurs BI (Kurs Tengah, log return)
7. Standardisasi zona waktu (WIB) & aturan *market cut-off* (T+1)
8. Penyelarasan hari libur/akhir pekan (*forward-rolling alignment*)
9. Agregasi berita harian (*group-by target date*)
10. Merge dataset final & ekspor


## 1. Setup & Instalasi Dependensi

In [126]:
# Jalankan sekali saja bila package belum tersedia di environment.
# Tanda seru (!) menjalankan perintah shell dari dalam notebook.
import sys
!{sys.executable} -m pip install -q --break-system-packages pandas numpy nltk Sastrawi langdetect holidays || \
{sys.executable} -m pip install -q pandas numpy nltk Sastrawi langdetect holidays



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [127]:
import re
import html
import warnings
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


## 2. Konfigurasi Pipeline

In [128]:
# Deteksi root proyek agar notebook tetap berjalan dari root proyek maupun folder src.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "cnbc-with-published.csv"
BI_PATH   = PROJECT_ROOT / "data" / "raw" / "bi-usd-rate.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Parameter filtering berita
# ------------------------------------------------------------------
MIN_WORD_COUNT = 20                     # ambang batas minimal panjang teks
TITLE_REPEAT_WEIGHT = 2                 # bobot pengulangan judul saat penggabungan teks

RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

# ------------------------------------------------------------------
# Parameter zona waktu & trading day
# ------------------------------------------------------------------
SOURCE_TZ = "America/New_York"
TARGET_TZ = "Asia/Jakarta"
MARKET_CUTOFF_HOUR = 15
START_DATE = pd.Timestamp("2021-09-01")
END_DATE = pd.Timestamp("2026-09-01 23:59:59")

print("Konfigurasi siap.")
print("NEWS_PATH:", NEWS_PATH)
print("BI_PATH:", BI_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Konfigurasi siap.
NEWS_PATH: d:\nlpproject-grestercinta\data\raw\cnbc-with-published.csv
BI_PATH: d:\nlpproject-grestercinta\data\raw\bi-usd-rate.csv
OUTPUT_DIR: d:\nlpproject-grestercinta\data\processed


## 3. Load Data Mentah && Selaraskan Format Data Raw

### 3.1 Load Data Mentah

In [129]:
news_raw = pd.read_csv(NEWS_PATH)
bi_raw = pd.read_csv(BI_PATH, header=None) if False else pd.read_csv(BI_PATH)

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (16733, 13)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets,2026-09-08T07:28:41+0000,2026-09-08,07:28 AM,0.0,2026-09-08T07:28:41+0000,ok
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call,NaN,NaN,NaN,NaN,NaN,not_found
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets,2026-09-08T04:02:43+0000,2026-09-08,04:02 AM,0.0,2026-09-08T04:02:43+0000,ok
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies,2026-09-08T04:17:07+0000,2026-09-08,04:17 AM,0.0,2026-09-08T04:17:07+0000,ok
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok


In [130]:
bi_raw.head(5)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal
0,1,1,17834.73,17657.27,9/1/2026 12:00:00 AM
1,2,1,17791.51,17614.49,8/31/2026 12:00:00 AM
2,3,1,17850.81,17673.19,8/28/2026 12:00:00 AM
3,4,1,17805.58,17628.42,8/27/2026 12:00:00 AM
4,5,1,17791.51,17614.49,8/26/2026 12:00:00 AM


### 3.2 Pisahkan Tanggal dan Jam pada dataset BI dan Geopolitik

In [131]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam dengan timezone WIB.
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["date_raw_dt_wib"] = (
    pd.to_datetime(news_raw["date_raw_dt"], errors="coerce", utc=True)
    .dt.tz_convert(TARGET_TZ)
    .dt.tz_localize(None)
)
news_raw["tanggal"] = news_raw["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt_wib"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt_wib"]].head(5)


Kolom Berita Geopolitik:


,title,tanggal,jam,date_raw_dt_wib
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,09/08/2026,09:28 PM,2026-09-08 21:28:41
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,08/01/2026,12:58 AM,2026-08-01 00:58:53
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",09/08/2026,06:02 PM,2026-09-08 18:02:43
3,"Yen hovers near seven-month high, dollar steadies",09/08/2026,06:17 PM,2026-09-08 18:17:07
4,Central banks are bringing gold reserves home asgeopoliticalrisks rise,06/17/2026,09:28 PM,2026-06-17 21:28:36


In [132]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


## 4. Filtering Data Berita Geopolitik

Tahapan:
1. **Deduplikasi** berdasarkan `url`, lalu berdasarkan kombinasi `title + tanggal + jam`.
2. **Filter teks kosong**: buang baris jika `title`, `preview_summary`, dan `full_text` semuanya kosong.
3. **Filter konten non-artikel**: buang item video, gambar, audio, podcast, galeri, dan live berdasarkan metadata URL/judul/section.
4. **Filter panjang teks**: buang berita dengan jumlah kata < `MIN_WORD_COUNT`.
5. **Filter rentang tanggal**: pertahankan berita dari 1 September 2021 sampai 1 September 2026.
6. **Filter relevansi kata kunci** dan section yang tidak relevan.

### 4.1 Deduplicate Dataset 

In [133]:
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_raw_dt"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 16733
Setelah filter rentang 1 September 2021 - 1 September 2026 : 8871
Setelah dedup url                  : 8871
Setelah dedup title+date_raw       : 7084


### 4.2 Filter Teks Kosong dan Konten Non-Artikel

In [134]:
# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 7084
Setelah drop video/gambar/audio      : 6214


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
8,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
9,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM
13,geopolitical risk,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks",https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN,Pro: Analysis,2026-05-18T05:37:18+0000,2026-05-18,05:37 AM,0.0,2026-05-18T05:37:18+0000,ok,2026-05-18 12:37:18,2026-05-18 19:37:18,05/18/2026,07:37 PM
15,geopolitical risk,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN,Pro: Income Investing,2026-04-10T19:25:17+0000,2026-04-10,07:25 PM,0.0,2026-04-10T19:25:17+0000,ok,2026-04-11 02:25:17,2026-04-11 09:25:17,04/11/2026,09:25 AM


### 4.3 Filter Panjang Teks Berita < 20 Kata

In [135]:
# Filter panjang teks minimal (gunakan full_text, fallback ke preview_summary)
effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= MIN_WORD_COUNT]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 6212


### 4.4 Filter Relefansi & Section Menggunakan Keyword

In [136]:
def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 6212
Setelah filter kategori/section     : 6196


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM


In [ ]:
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 1319


,url,title,preview_summary,full_text
3,https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks","Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN
4,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN
16,https://www.cnbc.com/2026/08/17/gold-prices-jeff-currie-gold.html?&qsearchterm=geopolitical risk,Veteran strategist Jeff Currie turns bullish on gold. Here’s why,"Gold investors are set for further volatility ahead, but the precious yellow metal is on an upward trajectory in the...",NaN
17,https://www.cnbc.com/2026/03/19/switzerland-considers-swiss-franc-intervention-amid-iran-war.html?&qsearchterm=geopo...,Switzerland risks the ire of the White House as it flags potential currency intervention,Swiss officials say the Iran war is making them more willing to intervene against any significant appreciation in th...,NaN
24,https://www.cnbc.com/2026/03/19/goldman-sees-risks-of-market-correction-rising-and-bonds-wont-help-weather-it.html?&...,Goldman sees risks of market correction rising — and bonds won’t help weather it,Goldman Sachs is warning investors to brace for a possible stock correction that won’t necessarily be buffered by bo...,NaN


## 5. Cleaning Data Kurs USD BI

Tahapan:
1. **Bersihkan metadata header/footer** — ambil hanya baris tabel riil dengan kolom `NO, Nilai, Kurs Jual, Kurs Beli, Tanggal` (fungsi dibuat generik agar tetap berfungsi meski file sumber memuat baris metadata di awal/akhir).
2. **Bersihkan karakter numerik** (pemisah ribuan) & konversi ke `float`.
3. **Filter outlier/anomali**: buang baris jika `Kurs Jual ≤ Kurs Beli`, atau bernilai nol/negatif.
4. Parse kolom `Tanggal` menjadi `datetime`.


In [138]:
EXPECTED_COLS = ["NO", "Nilai", "Kurs Jual", "Kurs Beli", "tanggal", "jam"]


def clean_bi_header_footer(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Ambil baris data BI dan abaikan metadata atau footer."""
    df = df_raw.copy()
    available_cols = [column for column in EXPECTED_COLS if column in df.columns]

    if len(available_cols) < len(EXPECTED_COLS):
        header_row_idx = None
        for index, row in df_raw.iterrows():
            row_values = {str(value).strip() for value in row.values}
            if set(EXPECTED_COLS).issubset(row_values):
                header_row_idx = index
                break

        if header_row_idx is not None:
            df = df_raw.iloc[header_row_idx + 1:].copy()
            df.columns = [str(value).strip() for value in df_raw.iloc[header_row_idx]]

    return df.dropna(how="all").reset_index(drop=True)


bi = clean_bi_header_footer(bi_raw)
bi = bi[[column for column in EXPECTED_COLS if column in bi.columns]].copy()

# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi["Tanggal"] = pd.to_datetime(
    bi["tanggal"].astype(str).str.strip() + " " + bi["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi.shape)
bi.head(3)

(1202, 7)


,NO,Nilai,Kurs Jual,Kurs Beli,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,08/28/2026,12:00 AM,2026-08-28


In [139]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


bi["Kurs Jual"] = clean_numeric(bi["Kurs Jual"])
bi["Kurs Beli"] = clean_numeric(bi["Kurs Beli"])
bi["NO"] = pd.to_numeric(bi["NO"], errors="coerce")
bi["Nilai"] = pd.to_numeric(bi["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi.loc[bi["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,09/03/2021,12:00 AM,2021-09-03


## 6. Text Preprocessing (NLP Pipeline)
